In [1]:
import fine as fn
import pyomo.environ as pyomo
import pandas as pd

In [2]:
def declareOpVarSet(self, esM, pyM):
    """
    Declare operation related sets (operation variables and mapping sets) in the pyomo object for a
    modeling class.

    :param esM: EnergySystemModel instance representing the energy system in which the component should be modeled.
    :type esM: EnergySystemModel instance

    :param pyM: pyomo ConcreteModel which stores the mathematical formulation of the model.
    :type pyM: pyomo ConcreteModel
    """
    compDict, abbrvName = self.componentsDict, self.abbrvName

    # Set for operation variables
    def declareOpVarSet(pyM):
        return (
            (loc, compName, ip)
            for compName, comp in compDict.items()
            for loc in comp.processedLocationalEligibility.index
            for ip in esM.investmentPeriods
            if comp.processedLocationalEligibility[loc] == 1
        )

    setattr(
        pyM,
        "operationVarSet_" + abbrvName,
        pyomo.Set(dimen=3, initialize=declareOpVarSet),
    )

    if self.dimension == "1dim":
        # Dictionary which lists all components of the modeling class at one location
        setattr(
            pyM,
            "operationVarDict_" + abbrvName,
            {
                ip: {
                    loc: {
                        compName
                        for compName in compDict
                        if (loc, compName, ip)
                        in getattr(pyM, "operationVarSet_" + abbrvName)
                    }
                    for loc in esM.locations
                }
                for ip in esM.investmentPeriods
            },
        )
    elif self.dimension == "2dim":
        # Dictionaries which list all outgoing and incoming components at a location
        setattr(
            pyM,
            "operationVarDictOut_" + abbrvName,
            {
                ip: {
                    loc: {
                        loc_: {
                            compName
                            for compName in compDict
                            if (loc + "_" + loc_, compName, ip)
                            in getattr(pyM, "operationVarSet_" + abbrvName)
                        }
                        for loc_ in esM.locations
                    }
                    for loc in esM.locations
                }
                for ip in esM.investmentPeriods
            },
        )
        setattr(
            pyM,
            "operationVarDictIn_" + abbrvName,
            {
                ip: {
                    loc: {
                        loc_: {
                            compName
                            for compName in compDict
                            if (loc_ + "_" + loc, compName, ip)
                            in getattr(pyM, "operationVarSet_" + abbrvName)
                        }
                        for loc_ in esM.locations
                    }
                    for loc in esM.locations
                }
                for ip in esM.investmentPeriods
            },
        )

def declareOpConstrSet5(self, pyM, constrSetName):
    """
    Declare operating mode set 5 for material consumption.
    This set is used for constraints where the operation (i.e. material consumption)
    is defined without the high-resolution time index 't'.
    Typically, the index is (loc, compName, mat, ip).
    """
    compDict, abbrvName = self.componentsDict, self.abbrvName
    varSet = getattr(pyM, "operationVarSet_" + abbrvName)

    def declareOpConstrSet5(pyM):
        # For each tuple (loc, compName, ip) in the existing operation variable set,
        # iterate over all materials for the given component.
        return (
            (loc, compName, mat, ip)
            for loc, compName, ip in varSet
            for mat in compDict[compName].material  # mat not accessed 
        )

    setattr(
        pyM,
        constrSetName + "5_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareOpConstrSet5)
    )

def declareMaterialVarSet(self, pyM, esM):
    compDict, abbrvName = self.componentsDict, self.abbrvName

    def declareMaterialSet(pyM):
        return (
            (loc, compName, mat, ip)
            for compName, comp in compDict.items()
            for loc in comp['processedLocationalEligibility'].keys()
            for mat in comp['materialIntensity'].keys() | comp['materialRecovery'].keys() 
            for ip in esM.investmentPeriods
        )

    setattr(
        pyM,
        "materialSet_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareMaterialSet),
    )

def declareMaterialVars(self, pyM, esM):

    """
    Declare variables for material intensity and recovery.

    :param pyM: Pyomo ConcreteModel which stores the mathematical formulation of the model.
    :type pyM: pyomo.ConcreteModel

    :param esM: Energy system model containing general information.
    :type esM: EnergySystemModel instance from the FINE package
    """

    abbrvName = self.abbrvName

    setattr(
        pyM,
        "materialIntensity_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

    setattr(
        pyM,
        "materialRecovery_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

def operationMaterialConsumption(
        self,
        pyM,
        esM,
        constrName,
        constrSetName,
        opVarName,
    ):
    """
    Define operation mode 5 for material sinks.

    This mode calculates the material consumption flow as:

        opVar[loc, comp, ip, p, t] = commisVar[loc, comp, ip] * materialIntensity

    where:
        - opVar is the sink's operation variable (interpreted as material consumption),
        - commisVar is the commissioning variable,
        - materialIntensity is a component attribute (accessed via materialIntensityName),
        - The constraint is applied over a set (e.g. a design set for material consumption indices)
        defined in pyM as: constrSetName + "5_" + abbrvName.

    This equality sets the consumption flow; then, a separate system-level constraint should ensure
    that the sum of material consumption across components does not exceed the available material supply.
    """
                                                                                                
    # call arguments
    compDict, abbrvName = self.componentsDict, self.abbrvName
    opVar = getattr(pyM, opVarName + "_" + abbrvName)
    commisVar = getattr(pyM, "commis_" + abbrvName)
    # right call of attributes (?) 
    materialIntensity = getattr(pyM, "materialIntensity_" + abbrvName)                        # add to op5 Set
    constrSet5 = getattr(pyM, constrSetName + "5_" + abbrvName)                               # constraint set 5 yet to be defined 

    def materialConsConstr(pyM, loc, compName, mat, ip):                                                       # for now include p for intra year variations
        # Get material intensity for given component
        # Enforce equality that defines material consumption
        return sum(opVar[loc, compName, ip, p, t] for p in esM.typicalPeriods for t in esM.hoursPerSegment) == commisVar[loc, compName, ip] * materialIntensity[loc, compName, mat, ip] # add material index
    
    setattr(
        pyM,
        constrName + "5_" + abbrvName,
        pyomo.Constraint(constrSet5, pyM.intraYearTimeSet, rule=materialConsConstr)
    )

In [3]:
#%% [markdown]
# ### Dummy Setup for Testing Material-Related Set and Variable Declarations

#%% [code]
import pyomo.environ as pyo
import pandas as pd

# Dummy Energy System Model (esm)
class DummyESM:
    def __init__(self):
        self.investmentPeriods = [2020, 2030]
        self.locations = ["A", "B"]
        # TypicalPeriods and hoursPerSegment are used to aggregate high-resolution time.
        self.typicalPeriods = [1, 2]  
        self.hoursPerSegment = [1, 1]  # For simplicity, each typical period is 1 hour

esm = DummyESM()

# Dummy component data
# For each component we define:
# - processedLocationalEligibility as a pandas Series with index = locations.
# - materialIntensity as a dict (e.g., {"steel": value, "copper": value}).
# - materialRecovery as a dict.
# - material as a list of material names.
comp1 = {
    "processedLocationalEligibility": pd.Series({"A": 1, "B": 0}),
    "materialIntensity": {"steel": 10, "copper": 5},
    "materialRecovery": {"steel": 0.8, "copper": 0.3},
    "material": ["steel", "copper"]
}

# For testing, we can have just one component.
compDict = {"comp1": comp1}

# Dummy Component Model class that holds our set/variable functions
class DummyComponentModel:
    def __init__(self, componentsDict, abbrvName, dimension="1dim"):
        self.componentsDict = componentsDict
        self.abbrvName = abbrvName
        self.dimension = dimension

    def declareOpVarSet(self, esM, pyM):
        compDict, abbrvName = self.componentsDict, self.abbrvName

        def declareOpVarSet_inner(pyM):
            return (
                (loc, compName, ip)
                for compName, comp in compDict.items()
                for loc in comp["processedLocationalEligibility"].index
                for ip in esM.investmentPeriods
                if comp["processedLocationalEligibility"][loc] == 1
            )

        setattr(
            pyM,
            "operationVarSet_" + abbrvName,
            pyo.Set(dimen=3, initialize=declareOpVarSet_inner)
        )
        return getattr(pyM, "operationVarSet_" + abbrvName)
    
    def declareOperationVars(
        self,
        pyM,
        esM,
        opVarName,
        opRateFixName="processedOperationRateFix",
        opRateMaxName="processedOperationRateMax",
        isOperationCommisYearDepending=False,
        flexibleConversion=False,
        relevanceThreshold=None,
    ):
        """
        Declare operation variables.

        The following operation modes are directly handled during variable creation as bounds instead of constraints.

        operation mode 4: If operationRateFix is given for components without a capacity variable,
        the variables are fixed with operationRateFix, i.e. the operation [commodityUnit*h] is equal to a time series.

        .. math::
            op^{comp,opType}_{loc,p,t} = \\text{opRateFix}^{comp,opType}_{loc,p,t}

        operation mode 5: If operationRateMax is given for components without a capacity variable,
        the variables are bounded by operationRateMax, i.e. the operation [commodityUnit*h] is limited by a time series.

        .. math::
            op^{comp,opType}_{loc,p,t} \\leq \\text{opRateMax}^{comp,opType}_{loc,p,t}

        :param pyM: pyomo ConcreteModel which stores the mathematical formulation of the model.
        :type pyM: pyomo ConcreteModel

        :param relevanceThreshold: Force operation parameters to be 0 if values are below the relevance threshold.
            |br| * the default value is None
        :type relevanceThreshold: float (>=0) or None

        :param isOperationCommisYearDepending: defines weather the operation variable is depending on the year
            of commissioning of the component. E.g. relevant if the commodity conversion, for example the efficiency,
            variates over the transformation pathway
        :type isOperationCommisYearDepending: str
        """
        abbrvName, compDict = self.abbrvName, self.componentsDict

        def opBounds(pyM, loc, compName, ip, p, t):
            if not getattr(compDict[compName], "hasCapacityVariable"):
                if not pyM.hasSegmentation:
                    if getattr(compDict[compName], opRateMaxName)[ip] is not None:
                        rate = getattr(compDict[compName], opRateMaxName)[ip]
                        if rate is not None:
                            if relevanceThreshold is not None:
                                validThreshold = 0 < relevanceThreshold
                                if validThreshold and (
                                    rate[loc][p, t] < relevanceThreshold
                                ):
                                    return (0, 0)
                            return (0, rate[loc][p, t])
                    elif getattr(compDict[compName], opRateFixName)[ip] is not None:
                        rate = getattr(compDict[compName], opRateFixName)[ip]
                        if rate is not None:
                            if relevanceThreshold is not None:
                                validThreshold = 0 < relevanceThreshold
                                if validThreshold and (
                                    rate[loc][p, t] < relevanceThreshold
                                ):
                                    return (0, 0)
                            return (rate[loc][p, t], rate[loc][p, t])
                    else:
                        return (0, None)
                elif getattr(compDict[compName], opRateMaxName)[ip] is not None:
                    rate = getattr(compDict[compName], opRateMaxName)[ip]
                    if rate is not None:
                        if relevanceThreshold is not None:
                            validThreshold = 0 < relevanceThreshold
                            if validThreshold and (
                                rate[loc][p, t] < relevanceThreshold
                            ):
                                return (0, 0)
                        return (
                            0,
                            rate[loc][p, t]
                            * esM.timeStepsPerSegment[ip].to_dict()[p, t],
                        )
                elif getattr(compDict[compName], opRateFixName)[ip] is not None:
                    rate = getattr(compDict[compName], opRateFixName)[ip]
                    if rate is not None:
                        if relevanceThreshold is not None:
                            validThreshold = 0 < relevanceThreshold
                            if validThreshold and (
                                rate[loc][p, t] < relevanceThreshold
                            ):
                                return (0, 0)
                        return (
                            rate[loc][p, t]
                            * esM.timeStepsPerSegment[ip].to_dict()[p, t],
                            rate[loc][p, t]
                            * esM.timeStepsPerSegment[ip].to_dict()[p, t],
                        )
                else:
                    return (0, None)
            else:
                return (0, None)

        if isOperationCommisYearDepending:
            # if the operation is depending on the year of commissioning, e.g. due to variable efficiencies over the
            # transformation pathway, the operation is additionally depending on commis
            def opBounds_commisDepending(pyM, loc, compName, commis, ip, p, t):
                return opBounds(pyM, loc, compName, ip, p, t)

            setattr(
                pyM,
                opVarName + "_" + abbrvName,
                pyomo.Var(
                    getattr(pyM, "operationCommisVarSet_" + abbrvName),
                    pyM.intraYearTimeSet,
                    domain=pyomo.NonNegativeReals,
                    bounds=opBounds_commisDepending,
                ),
            )
        elif flexibleConversion:
            setattr(
                pyM,
                opVarName + "_" + abbrvName,
                pyomo.Var(
                    getattr(pyM, "operationFlexVarSet_" + abbrvName),
                    pyM.intraYearTimeSet,
                    domain=pyomo.NonNegativeReals,
                ),
            )
        else:
            setattr(
                pyM,
                opVarName + "_" + abbrvName,
                pyomo.Var(
                    getattr(pyM, "operationVarSet_" + abbrvName),
                    pyM.intraYearTimeSet,
                    domain=pyomo.NonNegativeReals,
                    bounds=opBounds,
                ),
            )

    def declareOpConstrSet5(self, pyM, constrSetName):
        """
        Declare operating mode set 5 for material consumption.
        Typically, the index is (loc, compName, mat, ip).
        """
        compDict, abbrvName = self.componentsDict, self.abbrvName
        varSet = getattr(pyM, "operationVarSet_" + abbrvName)

        def declareOpConstrSet5_inner(pyM):
            # For each (loc, compName, ip) in varSet, iterate over all materials in comp['material']
            for loc, compName, ip in varSet:
                for mat in compDict[compName]["material"]:
                    yield (loc, compName, mat, ip)

        setattr(
            pyM,
            constrSetName + "5_" + abbrvName,
            pyo.Set(dimen=4, initialize=declareOpConstrSet5_inner)
        )
        return getattr(pyM, constrSetName + "5_" + abbrvName)

    def declareMaterialVarSet(self, pyM, esM):
        compDict, abbrvName = self.componentsDict, self.abbrvName

        def declareMaterialSet_inner(pyM):
            return (
                (loc, compName, mat, ip)
                for compName, comp in compDict.items()
                for loc in comp["processedLocationalEligibility"].keys()
                for mat in comp["materialIntensity"].keys() | comp["materialRecovery"].keys()
                for ip in esM.investmentPeriods
            )
        setattr(
            pyM,
            "materialSet_" + abbrvName,
            pyo.Set(dimen=4, initialize=declareMaterialSet_inner)
        )
        return getattr(pyM, "materialSet_" + abbrvName)

    def declareMaterialVars(self, pyM, esM):
        abbrvName = self.abbrvName
        # Create materialIntensity variable indexed by materialSet
        setattr(
            pyM,
            "materialIntensity_" + abbrvName,
            pyo.Var(
                getattr(pyM, "materialSet_" + abbrvName),
                domain=pyo.NonNegativeReals
            )
        )
        # Create materialRecovery variable indexed by materialSet
        setattr(
            pyM,
            "materialRecovery_" + abbrvName,
            pyo.Var(
                getattr(pyM, "materialSet_" + abbrvName),
                domain=pyo.NonNegativeReals
            )
        )
        return (getattr(pyM, "materialIntensity_" + abbrvName),
                getattr(pyM, "materialRecovery_" + abbrvName))

    def operationMaterialConsumption(self, pyM, esM, constrSetName, opVarName, constrName):
        """
        Define operation mode 5 for material sinks.
        The constraint:
            sum_{p,t} opVar[loc, comp, ip, p, t] = commisVar[loc, comp, ip] * materialIntensity[loc, comp, mat, ip]
        is applied over a set with indices (loc, comp, mat, ip).
        """
        compDict, abbrvName = self.componentsDict, self.abbrvName
        print("Vorhandene Attribute im Modell:", dir(pyM))  # Zeigt alle Attribute von pyM
        print("Gesuchter Name:", opVarName + "_" + abbrvName)
        print(f"opVarName: {opVarName}, abbrvName: {abbrvName}")
        ##################################################################################
        opVar = getattr(pyM, opVarName + "_" + abbrvName)##################################           
        ####################################################################################
        print(f"opVarName: {opVarName}, abbrvName: {abbrvName}")
        commisVar = getattr(pyM, "commis_" + abbrvName)
        materialIntensity = getattr(pyM, "materialIntensity_" + abbrvName)
        constrSet5 = getattr(pyM, constrSetName + "5_" + abbrvName)

        def materialConsConstr(pyM, loc, compName, mat, ip):
            total_op = sum(
                opVar[loc, compName, ip, p, t]
                for p in esM.typicalPeriods
                for t in esM.hoursPerSegment
            )
            # Use the material intensity for the given (loc, compName, mat, ip)
            return total_op == commisVar[loc, compName, ip] * materialIntensity[loc, compName, mat, ip]

        setattr(
            pyM,
            constrName + "5_" + abbrvName,
            pyo.Constraint(constrSet5, rule=materialConsConstr)
        )
        return getattr(pyM, constrName + "5_" + abbrvName)

#%% [code]
# Create a dummy Pyomo model
model = pyo.ConcreteModel()

# Create an instance of the DummyComponentModel with our dummy component dictionary
dummyCompModel = DummyComponentModel(compDict, abbrvName="dummy", dimension="1dim")

# For testing, we need to define also dummy variables for operation and commissioning.
# We simulate these by first declaring the operation variable set.
opVarSet = dummyCompModel.declareOpVarSet(esm, model)
print("Operation Variable Set:")
for item in opVarSet:
    print(item)

# Now, declare the operation constraint set for material consumption (set 5)
opConstrSet5 = dummyCompModel.declareOpConstrSet5(model, "opConstrSet")
print("\nOperation Constraint Set 5 (should be 4-tuples):")
for item in opConstrSet5:
    print(item)

# Next, declare the material variable set
matSet = dummyCompModel.declareMaterialVarSet(model, esm)
print("\nMaterial Set:")
for item in matSet:
    print(item)

# Now, declare material variables (materialIntensity and materialRecovery)
materialIntensityVar, materialRecoveryVar = dummyCompModel.declareMaterialVars(model, esm)
# For testing, we can fix some values to materialIntensity for a specific key.
# For example, fix materialIntensity for (loc="A", compName="comp1", mat="steel", ip=2020)
key_test = ("A", "comp1", "steel", 2020)
# We need to assign a value; since these are Pyomo Vars, we can fix them:
materialIntensityVar[key_test].fix(10)
materialRecoveryVar[key_test].fix(0.8)

# For the sake of testing the material consumption constraint,
# we need to also define dummy operation and commissioning variables.
# Let's create a dummy operation variable op_dummy, indexed over a 5-tuple: (loc, compName, ip, p, t)
def dummy_op_init(model, loc, compName, ip, p, t):
    # For simplicity, return a fixed value, e.g., 5.
    return 5

setattr(model, "op_dummy_dummy", pyo.Var(opVarSet, domain=pyo.NonNegativeReals, initialize=lambda model, loc, compName, ip: 5))
# However, note that our operationMaterialConsumption rule expects opVar to be indexed as (loc, compName, ip, p, t).
# We'll define a dummy variable over the expanded set:
def dummy_op_expanded_init(model, loc, compName, ip, p, t):
    return 5
# Create a dummy set for (loc, compName, ip, p, t):
def dummy_op_expanded_set(model):
    return (
        (loc, compName, ip, p, t)
        for loc, compName, ip in opVarSet
        for p in esm.typicalPeriods
        for t in esm.hoursPerSegment
    )

setattr(model, "op_dummy_expanded_set_dummy", pyo.Set(dimen=5, initialize=dummy_op_expanded_set))
setattr(model, "op_dummy_dummy", pyo.Var(getattr(model, "op_dummy_expanded_set_dummy"), domain=pyo.NonNegativeReals, initialize=dummy_op_expanded_init))

# And a dummy commissioning variable, indexed over opVarSet (which is (loc, compName, ip)):
def dummy_commis_init(model, loc, compName, ip):
    # For testing, set commissioning = 1 (component is commissioned)
    return 1

setattr(model, "commis_dummy", pyo.Var(opVarSet, domain=pyo.Binary, initialize=dummy_commis_init))

# Now, attach these dummy variables to the model so that the operationMaterialConsumption function can retrieve them.
# We'll assume opVarName = "op_dummy" and the commissioning variable is "commis_dummy".
# Next, call the operationMaterialConsumption function to create the constraint.
materialConsConstraint = dummyCompModel.operationMaterialConsumption(model, esm, "matConsOp", "opConstrSet5", "op_dummy")
print("\nMaterial Consumption Constraint:")
model.matConsOp5_dummy.pprint()

# Finally, to test the constraint, we could evaluate it for a specific key.
# For key ("A", "comp1", "steel", 2020), the rule is:
#    sum_{p,t} opVar[("A", "comp1", 2020, p, t)] == commisVar[("A", "comp1", 2020)] * materialIntensity[("A", "comp1", "steel", 2020)]
# Given our dummy values:
#   op_dummy is fixed at 5 for all (p,t), and we have two typicalPeriods and two hoursPerSegment, so the sum is 5*2*2 = 20.
#   commis_dummy for ("A", "comp1", 2020) is 1, and materialIntensity[("A", "comp1", "steel", 2020)] is fixed at 10.
#   The right-hand side would be 1 * 10 = 10.
# So the constraint would require 20 == 10, which will not hold.
# You might need to adjust the dummy values for a consistent test, or simply inspect the structure.


Operation Variable Set:
('A', 'comp1', 2020)
('A', 'comp1', 2030)

Operation Constraint Set 5 (should be 4-tuples):
('A', 'comp1', 'steel', 2020)
('A', 'comp1', 'copper', 2020)
('A', 'comp1', 'steel', 2030)
('A', 'comp1', 'copper', 2030)

Material Set:
('A', 'comp1', 'steel', 2020)
('A', 'comp1', 'steel', 2030)
('A', 'comp1', 'copper', 2020)
('A', 'comp1', 'copper', 2030)
('B', 'comp1', 'steel', 2020)
('B', 'comp1', 'steel', 2030)
('B', 'comp1', 'copper', 2020)
('B', 'comp1', 'copper', 2030)
(type=<class 'pyomo.core.base.var.IndexedVar'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
Vorhandene Attribute im Modell: ['Skip', '_BlockData__autoslot_mappers', '_Block_reserved_words', '_ComponentDataClass', '_DEFAULT_INDEX_CHECKING_ENABLED', '_PPRINT_INDENT', '__auto_slots__', '__autoslot_mappers__', '__class__', '__contains__'

AttributeError: 'ConcreteModel' object has no attribute 'matConsOp5_dummy'